In [12]:
import polars as pl
from pathlib import Path
# from regex_generator import words_to_compact_regex


src_file_path = Path("/workspaces/example/data/projects/ricu/inst/extdata/config/concept-dict.json")
src_df = pl.read_json(src_file_path)
d_items_df = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv.gz")
d_items_df = d_items_df.with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))
d_labitems_df = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz")
d_labitems_df = d_labitems_df.with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))

In [13]:
data = {}

In [14]:
# for concept in src_df.columns:
#     if (sources := src_df[concept][0].get("sources")) and sources.get("miiv"):
#         print(sources.get("miiv")[0].get("ids") is not None, sources.get("miiv")[0].get("callback") is not None, sources.get("miiv"))

In [15]:
# sources

In [29]:
excl_cnpt = {}

dict_keys(['outcome', 'demographics', 'medications', 'microbiology'])

In [30]:
for concept in src_df.columns:
    if (sources := src_df[concept][0].get("sources")) and sources.get("miiv"):
        cnpt_entry = src_df[concept][0]
        cnpt_category = cnpt_entry["category"].replace(" ", "_")

        if (ids := (miiv := sources["miiv"][0]).get("ids")) is None:
            if not excl_cnpt.get(cnpt_category):
                excl_cnpt[cnpt_category] = []
            excl_cnpt[cnpt_category].append(cnpt_entry)
            continue
        
        name = cnpt_entry["description"].replace(" ", "_").replace("/", "-")
        table = miiv["table"]

        if not data.get(cnpt_category):
            data[cnpt_category] = {}
        # print(cnpt_entry)

        if isinstance(ids, str) or (isinstance(ids, list) and isinstance(ids[0], str)):
            continue
        ids: list[str] = ids if isinstance(ids, list) else [ids]
        unit = cnpt_entry.get("unit")
        if isinstance(unit, list):
            unit = unit[0]
        
        data[cnpt_category][name] = {
            "name": name,
            "unit" : unit,
            "table" : table,
            "code" : None,
            "ids" : ids,
            "short" : concept,
            "min" : cnpt_entry.get("min"),
            "max" : cnpt_entry.get("max"), 
        }
        # print(data[cnpt_category][name])


        filtered_items = (d_labitems_df if table == "labevents" else d_items_df).filter(pl.col("itemid").is_in(ids)).select("code").collect()
        code_list = list(filtered_items.to_dict()["code"])
        regex = "|".join(code_list)
        # print(regex)
        # print(words_to_compact_regex(code_list))
        data[cnpt_category][name]["code"] = "(" + regex + ")"

In [38]:
for key in excl_cnpt.keys():
    print(len(excl_cnpt[key]), excl_cnpt[key])

3 [{'class': 'lgl_cncpt', 'description': 'in hospital mortality', 'omopid': 4306655, 'category': 'outcome', 'sources': {'aumc': [{'table': 'admissions', 'index_var': 'dateofdeath', 'val_var': 'dischargedat', 'callback': 'aumc_death', 'class': 'col_itm'}], 'eicu': [{'table': 'patient', 'index_var': 'unitdischargeoffset', 'val_var': 'hospitaldischargestatus', 'callback': "transform_fun(comp_na(`==`, 'Expired'))", 'class': 'col_itm'}], 'eicu_demo': [{'table': 'patient', 'index_var': 'unitdischargeoffset', 'val_var': 'hospitaldischargestatus', 'callback': "transform_fun(comp_na(`==`, 'Expired'))", 'class': 'col_itm'}], 'hirid': [{'ids': [110, 200], 'table': 'observations', 'sub_var': 'variableid', 'callback': 'hirid_death', 'class': 'hrd_itm'}], 'miiv': [{'table': 'admissions', 'index_var': 'deathtime', 'val_var': 'hospital_expire_flag', 'callback': 'transform_fun(comp_na(`==`, 1L))', 'class': 'col_itm'}], 'mimic': [{'table': 'admissions', 'index_var': 'deathtime', 'val_var': 'hospital_exp

In [17]:
data

{'respiratory': {'oxygen_saturation': {'name': 'oxygen_saturation',
   'unit': '%',
   'table': 'chartevents',
   'code': '(220227//Arterial O2 Saturation|220277//O2 saturation pulseoxymetry|226253//SpO2 Desat Limit)',
   'ids': [220277, 226253, 220227],
   'short': 'o2sat',
   'min': 50,
   'max': 100},
  'arterial_oxygen_saturation': {'name': 'arterial_oxygen_saturation',
   'unit': '%',
   'table': 'chartevents',
   'code': '(220227//Arterial O2 Saturation)',
   'ids': [220227],
   'short': 'sao2',
   'min': 50,
   'max': 100},
  'mechanical_ventilation_windows': {'name': 'mechanical_ventilation_windows',
   'unit': None,
   'table': 'procedureevents',
   'code': '(225792//Invasive Ventilation|225794//Non-invasive Ventilation)',
   'ids': [225792, 225794],
   'short': 'mech_vent',
   'min': None,
   'max': None},
  'respiratory_rate': {'name': 'respiratory_rate',
   'unit': 'insp/min',
   'table': 'chartevents',
   'code': '(220210//Respiratory Rate|224688//Respiratory Rate (Set)|22

In [18]:
base_path = Path("/workspaces/config/concept")


In [19]:
data

{'respiratory': {'oxygen_saturation': {'name': 'oxygen_saturation',
   'unit': '%',
   'table': 'chartevents',
   'code': '(220227//Arterial O2 Saturation|220277//O2 saturation pulseoxymetry|226253//SpO2 Desat Limit)',
   'ids': [220277, 226253, 220227],
   'short': 'o2sat',
   'min': 50,
   'max': 100},
  'arterial_oxygen_saturation': {'name': 'arterial_oxygen_saturation',
   'unit': '%',
   'table': 'chartevents',
   'code': '(220227//Arterial O2 Saturation)',
   'ids': [220227],
   'short': 'sao2',
   'min': 50,
   'max': 100},
  'mechanical_ventilation_windows': {'name': 'mechanical_ventilation_windows',
   'unit': None,
   'table': 'procedureevents',
   'code': '(225792//Invasive Ventilation|225794//Non-invasive Ventilation)',
   'ids': [225792, 225794],
   'short': 'mech_vent',
   'min': None,
   'max': None},
  'respiratory_rate': {'name': 'respiratory_rate',
   'unit': 'insp/min',
   'table': 'chartevents',
   'code': '(220210//Respiratory Rate|224688//Respiratory Rate (Set)|22

In [20]:
from pathlib import Path

def create(path: str, name: str, unit: str, table:str, cnpt_pattern: str) -> None:
    if unit == "%":
        unit = "\"%\""

    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    out_file = path / f"{name}.yml"

    # Existenz prüfen
    if out_file.exists():
        print(f"File already exists: {out_file}")
        return

    content = f"""name: {name}
version: 1.0.0
unit: {unit}

extension_columns:
  dataset: col("dataset")
  table: col("table")

mappings:
  - pattern:
      dataset: mimic-iv
      version: "3.1"
      table: {table}
      event: result
      code: {cnpt_pattern}
    columns:
      numeric_value: col(numeric_value)
      text_value: col(text_value)
"""

    with open(out_file, "w", encoding="utf8") as f:
        f.write(content)

    print(f"Created: {out_file}")

In [21]:
for category in data.keys():
    folder = base_path / category
    folder.mkdir(parents=True, exist_ok=True)
    for cnpt in (category := data[category]).keys() :
        print(category[cnpt])
        create(folder, category[cnpt]["name"], category[cnpt]["unit"], category[cnpt]["table"],category[cnpt]["code"])
    

{'name': 'oxygen_saturation', 'unit': '%', 'table': 'chartevents', 'code': '(220227//Arterial O2 Saturation|220277//O2 saturation pulseoxymetry|226253//SpO2 Desat Limit)', 'ids': [220277, 226253, 220227], 'short': 'o2sat', 'min': 50, 'max': 100}
Created: /workspaces/config/concept/respiratory/oxygen_saturation.yml
{'name': 'arterial_oxygen_saturation', 'unit': '%', 'table': 'chartevents', 'code': '(220227//Arterial O2 Saturation)', 'ids': [220227], 'short': 'sao2', 'min': 50, 'max': 100}
Created: /workspaces/config/concept/respiratory/arterial_oxygen_saturation.yml
{'name': 'mechanical_ventilation_windows', 'unit': None, 'table': 'procedureevents', 'code': '(225792//Invasive Ventilation|225794//Non-invasive Ventilation)', 'ids': [225792, 225794], 'short': 'mech_vent', 'min': None, 'max': None}
Created: /workspaces/config/concept/respiratory/mechanical_ventilation_windows.yml
{'name': 'respiratory_rate', 'unit': 'insp/min', 'table': 'chartevents', 'code': '(220210//Respiratory Rate|2246

In [22]:
data["chemistry"]["albumin"]

{'name': 'albumin',
 'unit': 'g/dL',
 'table': 'labevents',
 'code': '(50862//Albumin)',
 'ids': [50862],
 'short': 'alb',
 'min': 0,
 'max': 6}